In [ ]:
from pathlib import Path

PROJECT_SLUG = "07-f2-histognn"
WORK_ROOT = Path("/kaggle/working") / PROJECT_SLUG
PROJECT_STAGE = "synthetic core implemented; real artifact gate blocked"
BLOCKERS = [
    "no verified CellViT/TCGA-compatible artifact",
    "no approved real-data reader or live preflight entry point",
    "real data, GPU training, benchmark, and external services require approval",
]
print(f"STATUS: {PROJECT_SLUG}")
print(f"stage: {PROJECT_STAGE}")
print(f"work root: {WORK_ROOT}")
print("blockers:")
for blocker in BLOCKERS:
    print(f"- {blocker}")
print("next action: validate the staged source tree; no secrets or real data are accessed here.")


In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/07-f2-histognn")
REQUIRED_MARKERS = ("data", "models", "explanations", "training", "tests")
IMPORTANT_NAMES = {"README.md", "DESIGN.md", "requirements.txt", "pyproject.toml"}
candidates = []
if all((WORK_ROOT / marker).is_dir() for marker in REQUIRED_MARKERS):
    candidates.append(WORK_ROOT)
elif INPUT_ROOT.is_dir():
    for directory in [INPUT_ROOT, *sorted(path for path in INPUT_ROOT.rglob("*") if path.is_dir())]:
        if all((directory / marker).is_dir() for marker in REQUIRED_MARKERS):
            candidates.append(directory)
print("STATUS: staged source inspection completed without opening real-data contents.")
print("available top-level input directories:")
for path in sorted(path for path in INPUT_ROOT.iterdir() if path.is_dir()) if INPUT_ROOT.is_dir() else []:
    print(f"- {path}")
print("detected source-tree candidates:")
for path in candidates:
    print(f"- {path}")
for candidate in candidates:
    important = sorted(path for path in candidate.rglob("*") if path.is_file() and path.name in IMPORTANT_NAMES)
    print(f"important files under {candidate}:")
    for path in important:
        print(f"  - {path.relative_to(candidate)}")
if len(candidates) != 1:
    raise RuntimeError("FAIL: expected exactly one unambiguous source tree; do not guess a dataset path.")
print(f"source tree selected for the next cell: {candidates[0]}")
print("next action: copy this detected source tree after reviewing the printed candidate.")


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil

PROJECT_SLUG = "07-f2-histognn"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / PROJECT_SLUG
REQUIRED_MARKERS = ("data", "models", "explanations", "training", "tests")
candidates = [WORK_ROOT] if all((WORK_ROOT / marker).is_dir() for marker in REQUIRED_MARKERS) else []
if not candidates and INPUT_ROOT.is_dir():
    candidates = [path for path in [INPUT_ROOT, *sorted(path for path in INPUT_ROOT.rglob("*") if path.is_dir())] if all((path / marker).is_dir() for marker in REQUIRED_MARKERS)]
if len(candidates) != 1:
    raise RuntimeError("FAIL: source tree is missing or ambiguous; rerun Cell 2 and do not guess.")
source = candidates[0]
if source.resolve() == WORK_ROOT.resolve():
    print(f"STATUS: source is already staged at {WORK_ROOT}; no copy is required.")
elif WORK_ROOT.exists():
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    backup = WORK_ROOT.with_name(f"{PROJECT_SLUG}.backup.{stamp}")
    if backup.exists():
        raise RuntimeError(f"FAIL: backup path already exists: {backup}")
    shutil.move(str(WORK_ROOT), str(backup))
    print(f"preserved existing working directory at: {backup}")
if source.resolve() != WORK_ROOT.resolve():
    shutil.copytree(source, WORK_ROOT)
    print(f"STATUS: copied source tree from {source} to {WORK_ROOT}")
print("resulting directory structure:")
for path in sorted(WORK_ROOT.rglob("*")):
    if path.is_dir():
        print(f"[dir]  {path.relative_to(WORK_ROOT)}")
    elif path.is_file():
        print(f"[file] {path.relative_to(WORK_ROOT)}")
print("next action: validate the copied tree in Cell 4.")


In [ ]:
from pathlib import Path

WORK_ROOT = Path("/kaggle/working/07-f2-histognn")
REQUIRED_DIRECTORIES = ("data", "models", "explanations", "training", "tests", "notebooks")
REQUIRED_FILES = ("README.md", "DESIGN.md", "requirements.txt", "pyproject.toml")
if not WORK_ROOT.is_dir():
    raise RuntimeError(f"FAIL: missing working tree {WORK_ROOT}")
missing_directories = [name for name in REQUIRED_DIRECTORIES if not (WORK_ROOT / name).is_dir()]
missing_files = [name for name in REQUIRED_FILES if not (WORK_ROOT / name).is_file()]
requirements = WORK_ROOT / "requirements.txt"
notebook = WORK_ROOT / "notebooks" / "kaggle_run_07-f2-histognn.ipynb"
checks = [(f"directory:{name}", name not in missing_directories) for name in REQUIRED_DIRECTORIES]
checks += [(f"file:{name}", name not in missing_files) for name in REQUIRED_FILES]
checks += [("file:requirements.txt", requirements.is_file()), ("file:run notebook", notebook.is_file())]
for label, passed in checks:
    print(f"{'PASS' if passed else 'FAIL'} {label}")
if missing_directories or missing_files or not requirements.is_file() or not notebook.is_file():
    raise RuntimeError("FAIL: working-copy contract is incomplete; stop before installation.")
print("STATUS: working-copy structure passed.")
print("next action: install only from the exact project requirements file in Cell 5.")


In [ ]:
from pathlib import Path
import subprocess

WORK_ROOT = Path("/kaggle/working/07-f2-histognn")
REQUIREMENTS = WORK_ROOT / "requirements.txt"
if not REQUIREMENTS.is_file():
    raise RuntimeError("BLOCKED: this project has no approved requirements.txt; do not install ad hoc dependencies.")
print(f"STATUS: installing only {REQUIREMENTS}")
print("next action: review the complete pip output below; record conflict warnings in evidence.")
result = subprocess.run(["python", "-m", "pip", "install", "-r", str(REQUIREMENTS)], cwd=str(WORK_ROOT), check=False)
if result.returncode != 0:
    raise RuntimeError("FAIL: dependency installation failed; stop before restart or validation.")
print("STATUS: dependency installation exited successfully.")
print("next action: run Cell 6; restart only if Kaggle reports that installed packages require it.")


In [ ]:
from pathlib import Path

WORK_ROOT = Path("/kaggle/working/07-f2-histognn")
print("STATUS: kernel restart is not automatic.")
print("If Cell 5 changed an already-loaded package, restart the Kaggle kernel once from the Kaggle UI.")
print(f"working tree remains: {WORK_ROOT}")
print("next action after any restart: run Cell 7; otherwise continue directly to Cell 7.")


In [ ]:
import os

LIVE_PROVIDER_VARIABLES = ("WANDB_API_KEY", "HF_TOKEN", "KAGGLE_KEY", "KAGGLE_USERNAME", "HISTOGNN_PROVIDER_TOKEN")
removed = []
for name in LIVE_PROVIDER_VARIABLES:
    if name in os.environ:
        os.environ.pop(name)
        removed.append(name)
print("STATUS: provider-free synthetic process configuration applied.")
print(f"removed provider variable names: {removed if removed else 'none'}")
print("documented project-specific synthetic environment variables: none found; no new variables were invented.")
print("next action: run the documented offline synthetic command in Cell 8.")


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import os
import platform
import re
import resource
import subprocess
import torch

WORK_ROOT = Path("/kaggle/working/07-f2-histognn")
EVIDENCE_DIR = WORK_ROOT / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
text_path = EVIDENCE_DIR / f"synthetic_pytest_{stamp}.txt"
json_path = EVIDENCE_DIR / f"synthetic_validation_{stamp}.json"
env = os.environ.copy()
env["PYTHONPATH"] = str(WORK_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
command = ["python", "-m", "pytest", "-q", "-p", "no:cacheprovider"]
print(f"STATUS: running offline synthetic validation from {WORK_ROOT}")
print(f"command: PYTHONDONTWRITEBYTECODE=1 PYTHONPATH={WORK_ROOT} python -m pytest -q -p no:cacheprovider")
with text_path.open("w", encoding="utf-8") as evidence:
    result = subprocess.run(command, cwd=str(WORK_ROOT), env={**env, "PYTHONDONTWRITEBYTECODE": "1"}, stdout=evidence, stderr=subprocess.STDOUT, text=True, check=False)
summary = text_path.read_text(encoding="utf-8")
passed = re.search(r"(\d+) passed", summary)
warnings = re.search(r"(\d+) warnings?", summary)
digest = hashlib.sha256()
source_files = sorted([*WORK_ROOT.glob("data/*.py"), *WORK_ROOT.glob("models/*.py"), *WORK_ROOT.glob("explanations/*.py"), *WORK_ROOT.glob("training/*.py"), *WORK_ROOT.glob("tests/*.py"), WORK_ROOT / "requirements.txt", WORK_ROOT / "pyproject.toml"])
for path in source_files:
    digest.update(str(path.relative_to(WORK_ROOT)).encode("utf-8") + b"\0")
    digest.update(path.read_bytes())
record = {"status": "passed" if result.returncode == 0 else "failed", "timestamp_utc": stamp, "scope": "source-only synthetic contracts and graph smoke", "test_count": int(passed.group(1)) if passed else None, "warning_count": int(warnings.group(1)) if warnings else None, "failure_category": None if result.returncode == 0 else "synthetic_validation_failed", "text_evidence": str(text_path), "source_tree_sha256": digest.hexdigest(), "python": platform.python_version(), "numpy": version("numpy"), "torch": version("torch"), "torch_geometric": version("torch-geometric"), "pytest": version("pytest"), "cuda_available": torch.cuda.is_available(), "cuda_version": torch.version.cuda, "visible_gpu_count": torch.cuda.device_count(), "test_device": "cpu", "peak_memory_kib": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, "real_data_accessed": False, "gpu_training_run": False, "benchmark_metrics_emitted": False}
json_path.write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
if result.returncode != 0:
    print(f"FAIL: synthetic validation failed; sanitized evidence: {json_path}")
    raise RuntimeError("synthetic validation failed; stop before the next gate")
print("STATUS: synthetic validation passed.")
print(f"dated evidence path: {json_path}")
print("next action: inspect only the sanitized JSON in Cell 9.")


In [ ]:
from pathlib import Path
import json

EVIDENCE_DIR = Path("/kaggle/working/07-f2-histognn/evidence")
records = sorted(EVIDENCE_DIR.glob("synthetic_validation_*.json"), key=lambda path: path.stat().st_mtime)
if not records:
    raise RuntimeError("FAIL: no sanitized synthetic evidence JSON exists.")
record = json.loads(records[-1].read_text(encoding="utf-8"))
print("STATUS: latest sanitized synthetic evidence")
print(f"status: {record.get('status')}")
print(f"test count: {record.get('test_count')}")
print(f"warning count: {record.get('warning_count')}")
print(f"failure category: {record.get('failure_category')}")
print(f"timestamp UTC: {record.get('timestamp_utc')}")
if record.get("status") != "passed":
    raise RuntimeError("FAIL: synthetic validation did not pass; stop before any live gate.")
print("next action: proceed to the explicit approval message in Cell 10.")


In [ ]:
from pathlib import Path

EVIDENCE_DIR = Path("/kaggle/working/07-f2-histognn/evidence")
print("Synthetic validation passed. Live provider/data/model access is still blocked until explicitly approved.")
print(f"evidence directory: {EVIDENCE_DIR}")
print("next action: review the runbook and obtain explicit approval before considering Cell 12.")


In [ ]:
print("STATUS: separate secret-loading cell; not executed automatically.")
SECRET_NAME = None
if SECRET_NAME is None:
    print("secret loaded: False")
    print("next action: do not continue; no approved Kaggle Secret name is documented for this project.")
else:
    from kaggle_secrets import UserSecretsClient
    secret = UserSecretsClient().get_secret(SECRET_NAME)
    print(f"secret loaded: {bool(secret)}")
    print("next action: keep the secret in memory only; never print or write it to evidence.")


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json

WORK_ROOT = Path("/kaggle/working/07-f2-histognn")
EVIDENCE_DIR = WORK_ROOT / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
APPROVED = False  # EXPLICIT APPROVAL REQUIRED: set True only after written approval for a named artifact.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
evidence = EVIDENCE_DIR / f"real_artifact_preflight_{stamp}.json"
record = {"status": "blocked", "timestamp_utc": stamp, "failure_category": "explicit_approval_required", "artifact_accessed": False}
evidence.write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
print("STATUS: bounded live preflight is blocked.")
print("required approval: written approval naming the real artifact and authorizing rights, checksum, schema, provenance, and group-isolation checks.")
print(f"sanitized evidence path: {evidence}")
if not APPROVED:
    print("next action: stop; do not read an artifact, load a provider, or run a probe.")
else:
    raise RuntimeError("BLOCKED: no documented live preflight entry point exists in this project tree; do not improvise one.")


In [ ]:
from pathlib import Path

WORK_ROOT = Path("/kaggle/working/07-f2-histognn")
APPROVED = False  # EXPLICIT APPROVAL REQUIRED before any training, benchmark, or evaluation command.
COMMAND_LABEL = "training/full benchmark/evaluation: no documented command available"
print(f"STATUS: {COMMAND_LABEL}")
print("No benchmark or training entry point is present in the audited project files.")
print("No command is executed automatically, and no overnight work is started.")
if not APPROVED:
    print("next action: stop; obtain an approved command and explicit approval after the real preflight passes.")
else:
    raise RuntimeError("BLOCKED: command is intentionally absent until an approved benchmark/training entry point is documented.")
